# DACN — checkpoint Gaussian batch8 epoch20 → private validation pairs → CPU diagnostic
This notebook performs inference only, never training or held-out-test evaluation.
Select a GPU runtime you can access. Availability/cost depends on your Colab account.
Use your own HF read access; enter token through getpass, never paste it in a cell.
No patient images or sample UUIDs are displayed. NPZ remains private in this runtime;
only download/share it privately with authorized project members. Runtime can reset,
so save artifacts before disconnecting. Metrics are exploratory validation candidates.


In [ ]:
import subprocess
import sys
from pathlib import Path

SOURCE_REVISION = "7a0f024d8a0027dfdd90d3eb3d45b4381105a85f"
REPO = Path("/content/DACN")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/tanh1c/DACN.git", str(REPO)], check=True)
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("Existing checkout has local changes; use a fresh Colab runtime")
subprocess.run(["git", "fetch", "origin", SOURCE_REVISION], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO, check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[train,hub,data]"], check=True
)
sys.path.insert(0, str(REPO / "src"))

In [ ]:
import hashlib
import json
from getpass import getpass

from huggingface_hub import HfApi, hf_hub_download, snapshot_download

hf_token = getpass("HF token with read access to private DACN repos: ")
api = HfApi(token=hf_token)
api.whoami()
DATA_REV = "350c74f2313e4e73ca65e7ffc6e9dcf74040b3b7"
MODEL_REV = "1788b1232aa19b3c9ae96aa95a74e2975f4b592a"
META_SHA = "9c60fac0ff2a2d6274e82db466e5e338303d0dc24986b3bab4b32b6e710af218"
CKPT_SHA = "b4fc177db493fce2143a6412161068c7af73a25e8ae7ef9d88c117863c7e37f1"
DATA_REPO = "dacn-bmad-team/bmad-flair-patient-disjoint-v1"
MODEL_REPO = "dacn-bmad-team/dacn-runs"
checkpoint_path = hf_hub_download(
    MODEL_REPO,
    "runs/ddpm-partial-t4-b8-e30-cont-20261102T034644Z/ddpm_epoch_020.pt",
    revision=MODEL_REV,
    token=hf_token,
)
checkpoint_bytes = Path(checkpoint_path).read_bytes()
assert hashlib.sha256(checkpoint_bytes).hexdigest() == CKPT_SHA
DATA = Path("/content/private_dacn_validation")
snapshot_download(
    DATA_REPO,
    repo_type="dataset",
    revision=DATA_REV,
    local_dir=str(DATA),
    token=hf_token,
    allow_patterns=["dataset_metadata.json", "data/val/*"],
)
from dacn.evaluation.artifacts import _validation_data

_validation_data(DATA, META_SHA)
del hf_token
print("Pinned checkpoint and validation checksums verified; no test downloaded.")

## Explicit inference step
Set RUN_INFERENCE=True when ready. This processes all83 validation slices at t*=300
with sampling seed17 and records elapsed time. It can take time; it does not spend
Modal credits. Avoid leaving a paid Colab runtime running unattended. No automatic
retries. A failure is recorded locally; partial output is not a completed result.


In [ ]:
RUN_INFERENCE = False
if not RUN_INFERENCE:
    raise RuntimeError("Set RUN_INFERENCE=True to explicitly run validation inference")

import io
import time
import uuid
from importlib.metadata import version

import numpy as np
import torch

from dacn.evaluation.ddpm import _ddpm_backend, produce_validation_artifact

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required for this full-cohort inference workflow")
state = torch.load(io.BytesIO(checkpoint_bytes), map_location="cpu", weights_only=False)
assert state["epoch"] == 20 and state["optimizer_steps"] == 18760 and state["config"]["seed"] == 17
OUT = Path("/content/private_dacn_runs") / ("gaussian-e20-t300-" + uuid.uuid4().hex[:12])
OUT.mkdir(parents=True, exist_ok=False)
artifact = OUT / "validation_pairs.npz"
provenance = {
    "dataset_repo_id": DATA_REPO,
    "dataset_revision": DATA_REV,
    "dataset_metadata_sha256": META_SHA,
    "model_repo_id": MODEL_REPO,
    "model_revision": MODEL_REV,
    "checkpoint_sha256": CKPT_SHA,
    "code_revision": SOURCE_REVISION,
    "train_seed": state["config"]["seed"],
    "sampling_seed": 17,
    "starting_timestep": 300,
}
(OUT / "provenance.json").write_text(json.dumps(provenance, indent=2))
manifest = {
    "status": "started",
    "scope": "full validation inference; test sealed",
    "provenance": provenance,
    "gpu": torch.cuda.get_device_name(0),
    "packages": {name: version(name) for name in ["torch", "numpy", "scipy", "scikit-learn"]},
}
(OUT / "inference_manifest.json").write_text(json.dumps(manifest, indent=2))
started = time.perf_counter()
try:
    adapter = _ddpm_backend(checkpoint_bytes, state["config"], "cuda")
    by_id, images, _, _ = _validation_data(DATA, META_SHA)
    assert len(by_id) == 83
    for sample_id, position in list(by_id.items())[:4]:
        output = adapter(
            images[position][None, None],
            sample_id=sample_id,
            starting_timestep=3,
            sampling_base_seed=17,
        )
        assert output.shape == (1, 1, 256, 256) and np.isfinite(output).all()
    manifest["smoke_samples"] = 4
    deadline = started + 45 * 60

    def bounded_adapter(*args, **kwargs):
        if time.perf_counter() >= deadline:
            raise TimeoutError("Validation exceeded the 45-minute budget")
        result = adapter(*args, **kwargs)
        if time.perf_counter() >= deadline:
            raise TimeoutError("Validation exceeded the 45-minute budget")
        return result
    receipt = produce_validation_artifact(DATA, artifact, provenance, bounded_adapter)
    manifest.update(status="completed", receipt=receipt)
except Exception as exc:
    manifest.update(status="failed", error_class=type(exc).__name__)
    raise
finally:
    manifest["elapsed_seconds"] = time.perf_counter() - started
    (OUT / "inference_manifest.json").write_text(json.dumps(manifest, indent=2))
print({"status": manifest["status"], "samples": receipt["samples"]})

In [ ]:
from dacn.evaluation.diagnostics import compare

assert (
    hashlib.sha256(artifact.read_bytes()).hexdigest() == receipt["reconstruction_artifact_sha256"]
)
report = compare(artifact, DATA, provenance)
(OUT / "diagnostic_report.json").write_text(json.dumps(report, indent=2, allow_nan=False))
for method, variants in report["results"].items():
    print(
        method,
        {
            variant: {
                "pixel_AP": variants[variant]["metrics"]["pixel"]["average_precision"],
                "pixel_Dice": variants[variant]["metrics"]["pixel"]["dice_micro"],
                "image_AUROC": variants[variant]["metrics"]["image"]["auroc"],
            }
            for variant in ["baseline", "foreground_floor"]
        },
    )
print("Report saved. Thresholds are not frozen; do not open test.")

## Private handoff
Download the aggregate diagnostic report, provenance and inference manifest.
The separate NPZ contains medical-derived arrays and private UUIDs; download it
only to a private project folder, never attach it to GitHub or Linear. Share
through an authorized private channel. Add actual run status/source/environment
and aggregate evidence to RESULT_TRACKING; do not report this notebook as run
until the inference cell has really completed.


In [ ]:
from google.colab import files

files.download(str(OUT / "diagnostic_report.json"))
files.download(str(OUT / "provenance.json"))
files.download(str(OUT / "inference_manifest.json"))
# Explicit private download if needed for B's local CPU analysis:
# files.download(str(artifact))